# Ultrasonic Sensor Library Documentation

## Overview
The **Ultrasonic** library provides a comprehensive interface for HC-SR04 compatible ultrasonic distance sensors on Raspberry Pi. This library supports multiple sensors with advanced features like distance averaging, timeout handling, and proper GPIO management.

**Full source code available at [Ultrasonic_sens.py](Ultrasonic_sens.py)**

## Libraries Used
- **RPi.GPIO**: GPIO control for Raspberry Pi
- **time**: Precise timing for pulse measurements

## Let's Start Coding !  
### 1. Import All the Libraries  

Import the required libraries for IR sensor control and timing operations.  

- **RPi.GPIO** → Provides access to Raspberry Pi GPIO pins for sensor input/output  
- **time** → Provides delay functions for stable sensor readings  

In [22]:
import RPi.GPIO as GPIO
import time

## 2. Class: `Ultrasonic`

This class encapsulates the logic for ultrasonic distance measurement using GPIO.  
It is designed for robust operation and ensures safe initialization and cleanup.

- **Class Name**: `Ultrasonic`

- **Class Variables**:
  - `__init_check` → Internal flag to prevent multiple GPIO initializations  
  - `SOUND_SPEED` *(float = 34300)* → Speed of sound in cm/s at 20°C, used for distance calculation  


In [23]:
class Ultrasonic:
    """
    Class to represent an ultrasonic sensor.
    """
    __init_check = False 
    SOUND_SPEED = 34300  # Speed of sound in cm/s

### 3.1 The Constructor `__init__(self, Left_sensor=5, Front_sensor=16, Right_sensor=18, debug=False)`

This method initializes the ultrasonic sensor system with three sensors (Left, Front, Right).  
Each sensor uses a single GPIO pin for both trigger and echo signals.

- **Function Name**: `__init__`

- **Parameters**:  
  - `Left_sensor` *(int, default=5)* → GPIO pin for the left ultrasonic sensor  
  - `Front_sensor` *(int, default=16)* → GPIO pin for the front ultrasonic sensor  
  - `Right_sensor` *(int, default=18)* → GPIO pin for the right ultrasonic sensor  
  - `debug` *(bool, default=False)* → Enable debug messages  

- **Usage**:  
  - Ensures `GPIO.setmode(GPIO.BCM)` is initialized only once using `__init_check`  
  - Disables GPIO warnings for cleaner terminal output  
  - Stores pin assignments in the instance for later distance measurement  
  - If `debug=True`, prints initialization details including pin mappings  


In [ ]:
def __init__(self, Left_sensor=5, Front_sensor=16, Right_sensor=18, debug=False):
        """
        Initializes GPIO pins for the ultrasonic sensors.
        :param Left_sensor: Left sensor GPIO pin (default 5)
        :param Front_sensor: Front sensor GPIO pin (default 16) 
        :param Right_sensor: Right sensor GPIO pin (default 18)
        :param debug: Enable debug mode (default False)
        """
        if not Ultrasonic.__init_check:
            self.Left_sensor = Left_sensor
            self.Front_sensor = Front_sensor  
            self.Right_sensor = Right_sensor
            self.debug = debug

            # Initialize GPIO mode only once
            GPIO.setmode(GPIO.BCM)
            GPIO.setwarnings(False)  # Disable warnings for cleaner output
            
            if self.debug:
                print("Ultrasonic sensor system initialized.")
                print(f"Sensors: Left={Left_sensor}, Front={Front_sensor}, Right={Right_sensor}")
            
            Ultrasonic.__init_check = True
        else:
            if self.debug: 
                print("Ultrasonic sensor already initialized.")

### 3.2 The Method `send_trigger_pulse(self, pin)`

This method sends a **10 µs trigger pulse** on the specified ultrasonic sensor pin.

- **Function Name**: `send_trigger_pulse`

- **Parameters**:  
  - `pin` *(int)* → GPIO pin number connected to the ultrasonic sensor (used for both trigger and echo)  

- **Usage**:  
  - Configures the given pin as an output (`GPIO.OUT`)  
    ```python
    GPIO.setup(pin, GPIO.OUT)
    ```
  - Sends a **2 µs LOW signal** to stabilize the sensor  
    ```python
    GPIO.output(pin, False)
    time.sleep(0.000002)  # 2 microseconds low
    ```
  - Sends a **10 µs HIGH signal** as the trigger pulse  
    ```python
    GPIO.output(pin, True)
    time.sleep(0.00001)   # 10 microseconds high (trigger pulse)
    ```
  - Pulls the pin LOW again to complete the pulse  
    ```python
    GPIO.output(pin, False)
    ```
  - If `debug=True`, prints confirmation with the pin number  
    ```python
    if self.debug:
        print(f"Trigger pulse sent to pin {pin}")
    ```


In [ ]:
    def send_trigger_pulse(self, pin):
        """
        Sends a proper 10 microsecond trigger pulse on the specified pin.
        """
        GPIO.setup(pin, GPIO.OUT)
        GPIO.output(pin, False)
        time.sleep(0.000002)  # 2 microseconds low
        GPIO.output(pin, True)
        time.sleep(0.00001)   # 10 microseconds high (trigger pulse)
        GPIO.output(pin, False)
        
        if self.debug:
            print(f"Trigger pulse sent to pin {pin}")

### 3.3 The Method `wait_for_echo(self, pin)`

This method waits for the ultrasonic sensor’s echo response and measures the pulse duration.  
It uses timeout checks to prevent infinite waiting when no echo is received.  

- **Function Name**: `wait_for_echo`

- **Parameters**:  
  - `pin` *(int)* → GPIO pin number connected to the ultrasonic sensor (used for both trigger and echo)  

- **Usage**:  
  - Configure the pin as input:  
    ```python
    GPIO.setup(pin, GPIO.IN)
    ```
  - Define timeout values:  
    ```python
    timeout_start = 0.02    # 20ms timeout for echo start
    timeout_duration = 0.03 # 30ms timeout for echo duration (~5m range)
    ```
  - Wait for echo to start (rising edge):  
    ```python
    while GPIO.input(pin) == 0:
        if time.time() - start_time > timeout_start:
            if self.debug:
                print(f"Timeout waiting for echo start on pin {pin}")
            return None
    ```
  - Record start time, then wait for echo to end (falling edge):  
    ```python
    pulse_start = time.time()
    pulse_end = pulse_start
    while GPIO.input(pin) == 1:
        pulse_end = time.time()
        if pulse_end - pulse_start > timeout_duration:
            if self.debug:
                print(f"Timeout during echo on pin {pin}")
            return None
    ```
  - Calculate pulse duration:  
    ```python
    pulse_duration = pulse_end - pulse_start
    ```
  - If `debug=True`, print the measured duration in microseconds.  
  - Return the pulse duration (seconds).  


In [ ]:
 def wait_for_echo(self, pin):
        """
        Waits for the echo response and measures the pulse duration.
        Returns pulse duration in seconds, or None if timeout.
        """
        GPIO.setup(pin, GPIO.IN)
        
        # Timeout values
        timeout_start = 0.02    # 20ms timeout for echo start
        timeout_duration = 0.03 # 30ms timeout for echo duration (max ~5m range)
        
        start_time = time.time()
        
        # Wait for echo to start (rising edge)
        while GPIO.input(pin) == 0:
            if time.time() - start_time > timeout_start:
                if self.debug:
                    print(f"Timeout waiting for echo start on pin {pin}")
                return None

        # Record when echo started
        pulse_start = time.time()
        pulse_end = pulse_start

        # Wait for echo to end (falling edge)
        while GPIO.input(pin) == 1:
            pulse_end = time.time()
            if pulse_end - pulse_start > timeout_duration:
                if self.debug:
                    print(f"Timeout during echo on pin {pin}")
                return None
        
        # Calculate pulse duration
        pulse_duration = pulse_end - pulse_start
        
        if self.debug:
            print(f"Echo duration: {pulse_duration*1000000:.1f} microseconds")
            
        return pulse_duration


### 3.4 The Method `get_distance(self, pin)`

This method measures distance using the ultrasonic sensor on the specified pin.  
It sends a trigger pulse, waits for the echo, and calculates the distance in centimeters.  

- **Function Name**: `get_distance`

- **Parameters**:  
  - `pin` *(int)* → GPIO pin number connected to the ultrasonic sensor (used for both trigger and echo)  

- **Usage**:  
  - Send the trigger pulse:  
    ```python
    self.send_trigger_pulse(pin)
    time.sleep(0.00001)  # Ensure trigger is processed
    ```
  - Wait for and measure the echo:  
    ```python
    pulse_duration = self.wait_for_echo(pin)
    if pulse_duration is None:
        if self.debug:
            print(f"No valid echo received from pin {pin}")
        return None
    ```
  - Calculate distance in centimeters:  
    ```python
    distance = (pulse_duration * self.SOUND_SPEED) / 2
    ```
    *(Divide by 2 because sound travels to the object and back.)*  
  - Validate the distance within HC-SR04’s usable range (2–400 cm):  
    ```python
    if distance < 2 or distance > 400:
        if self.debug:
            print(f"Distance out of valid range: {distance:.1f}cm")
        return None
    ```
  - If `debug=True`, print the measured distance:  
    ```python
    print(f"Pin {pin}: {distance:.1f}cm")
    ```
  - Return the valid distance value in **centimeters**.  


In [ ]:
 def get_distance(self, pin):
        """
        Measures distance using the ultrasonic sensor on the specified pin.
        Returns distance in centimeters, or None if measurement failed.
        """
        try:
            # Send trigger pulse
            self.send_trigger_pulse(pin)
            
            # Small delay to ensure trigger is processed
            time.sleep(0.00001)
            
            # Wait for and measure echo
            pulse_duration = self.wait_for_echo(pin)
            
            if pulse_duration is None:
                if self.debug:
                    print(f"No valid echo received from pin {pin}")
                return None
            
            # Calculate distance: distance = (time * speed) / 2
            # Divide by 2 because sound travels to object and back
            distance = (pulse_duration * self.SOUND_SPEED) / 2
            
            # Validate distance (HC-SR04 range: 2cm to 400cm)
            if distance < 2 or distance > 400:
                if self.debug:
                    print(f"Distance out of valid range: {distance:.1f}cm")
                return None
                
            if self.debug:
                print(f"Pin {pin}: {distance:.1f}cm")
                
            return distance
            
        except Exception as e:
            if self.debug:
                print(f"Error measuring distance on pin {pin}: {e}")
            return None

### 3.5 The Method `get_distance_average(self, pin, samples=3, delay=0.1)`

This method takes multiple distance measurements from the ultrasonic sensor and returns the **average value** for improved accuracy.  

- **Function Name**: `get_distance_average`

- **Parameters**:  
  - `pin` *(int)* → GPIO pin number connected to the ultrasonic sensor.  
  - `samples` *(int, default=3)* → Number of distance readings to take.  
  - `delay` *(float, default=0.1)* → Delay (in seconds) between consecutive readings.  

- **Usage**:  
  - Collect multiple readings:  
    ```python
    readings = []
    for i in range(samples):
        distance = self.get_distance(pin)
        if distance is not None:
            readings.append(distance)
        if i < samples - 1:  # Skip delay after last sample
            time.sleep(delay)
    ```
  - If no valid readings are collected, return `None`:  
    ```python
    if not readings:
        if self.debug:
            print(f"No valid readings from pin {pin}")
        return None
    ```
  - Compute the average distance:  
    ```python
    average = sum(readings) / len(readings)
    ```
  - If `debug=True`, print the average result with sample count:  
    ```python
    print(f"Pin {pin} average from {len(readings)} samples: {average:.1f}cm")
    ```
  - Return the average distance in **centimeters**.  


In [ ]:
def get_distance_average(self, pin, samples=3, delay=0.1):
        """
        Gets multiple distance readings and returns the average for better accuracy.
        :param pin: GPIO pin number
        :param samples: Number of samples to take (default 3)
        :param delay: Delay between samples in seconds (default 0.1)
        :return: Average distance in cm, or None if all samples failed
        """
        readings = []
        
        for i in range(samples):
            distance = self.get_distance(pin)
            if distance is not None:
                readings.append(distance)
            
            if i < samples - 1:  # Don't delay after last sample
                time.sleep(delay)
        
        if not readings:
            if self.debug:
                print(f"No valid readings from pin {pin}")
            return None
            
        average = sum(readings) / len(readings)
        
        if self.debug:
            print(f"Pin {pin} average from {len(readings)} samples: {average:.1f}cm")
            
        return average

### 3.6 The Method `distances(self, use_average=False, samples=3)`

This method retrieves distance measurements from all three ultrasonic sensors (**Left, Front, Right**) and returns them as a tuple.  

- **Function Name**: `distances`

- **Parameters**:  
  - `use_average` *(bool, default=False)* → If `True`, uses multiple readings per sensor and returns the averaged value.  
  - `samples` *(int, default=3)* → Number of readings to average (only applies if `use_average=True`).  

- **Usage**:  
  - If averaging is enabled, call `get_distance_average` for each sensor:  
    ```python
    Left = self.get_distance_average(self.Left_sensor, samples)
    Front = self.get_distance_average(self.Front_sensor, samples)
    Right = self.get_distance_average(self.Right_sensor, samples)
    ```
  - Otherwise, take a single distance reading per sensor with small delays between them:  
    ```python
    Left = self.get_distance(self.Left_sensor)
    time.sleep(0.1)
    Front = self.get_distance(self.Front_sensor)
    time.sleep(0.1)
    Right = self.get_distance(self.Right_sensor)
    ```
  - If `debug=True`, print all sensor values:  
    ```python
    print(f"All sensors - Left: {Left}, Front: {Front}, Right: {Right}")
    ```
  - Return the results as a tuple:  
    ```python
    return Left, Front, Right
    ```

- **Returns**:  
  - `tuple(float or None, float or None, float or None)` → Distances for `(Left, Front, Right)` in centimeters.  


In [ ]:
def distances(self, use_average=False, samples=3):
        """
        Get distance measurements from all three sensors.
        :param use_average: If True, uses averaged readings for better accuracy
        :param samples: Number of samples for averaging (if use_average=True)
        :return: Tuple of (Left, Front, Right) distances in cm
        """
        if use_average:
            Left = self.get_distance_average(self.Left_sensor, samples)
            Front = self.get_distance_average(self.Front_sensor, samples)
            Right = self.get_distance_average(self.Right_sensor, samples)
        else:
            Left = self.get_distance(self.Left_sensor)
            time.sleep(0.1)  # Small delay between sensors
            Front = self.get_distance(self.Front_sensor)
            time.sleep(0.1)
            Right = self.get_distance(self.Right_sensor)
            
        if self.debug:
            print(f"All sensors - Left: {Left}, Front: {Front}, Right: {Right}")
            
        return Left, Front, Right


### 3.7 The Method `get_closest_obstacle(self)`

This method determines the nearest obstacle detected by the ultrasonic sensors and identifies its direction.  

- **Function Name**: `get_closest_obstacle`  

- **Parameters**:  
  - *(None)* → This method does not take external parameters; it uses the sensor readings from `distances()`.  

- **Usage**:  
  - Collect distance values from all sensors:  
    ```python
    left, front, right = self.distances()
    ```
  - Build a dictionary of valid (non-None) readings:  
    ```python
    valid_readings = {}
    if left is not None:
        valid_readings['Left'] = left
    if front is not None:
        valid_readings['Front'] = front
    if right is not None:
        valid_readings['Right'] = right
    ```
  - If no valid readings are available, return:  
    ```python
    return None, None
    ```
  - Find the closest obstacle by selecting the minimum distance:  
    ```python
    closest_direction, closest_distance = min(
        valid_readings.items(), key=lambda kv: kv[1]
    )
    ```
  - If `debug=True`, print the closest obstacle details:  
    ```python
    print(f"Closest obstacle: {closest_distance:.1f}cm to the {closest_direction}")
    ```
  - Return the result as a tuple:  
    ```python
    return closest_distance, closest_direction
    ```

- **Returns**:  
  - `tuple(float, str)` → Distance in cm and direction (`"Left"`, `"Front"`, `"Right"`)  
  - `(None, None)` → If no obstacles are detected  


In [ ]:
def get_closest_obstacle(self):
        """
        Returns the distance to the closest obstacle and its direction.
        :return: Tuple of (distance, direction) or (None, None) if no obstacles detected
        """
        left, front, right = self.distances()
        
        # Filter out None values
        valid_readings = {}
        if left is not None:
            valid_readings['Left'] = left
        if front is not None:
            valid_readings['Front'] = front  
        if right is not None:
            valid_readings['Right'] = right
            
        if not valid_readings:
            return None, None
            
        # Find closest obstacle (type-checker friendly)
        closest_direction, closest_distance = min(
            valid_readings.items(), key=lambda kv: kv[1]
        )
        
        if self.debug:
            print(f"Closest obstacle: {closest_distance:.1f}cm to the {closest_direction}")
            
        return closest_distance, closest_direction

### 3.8 The Method `is_path_clear(self, min_distance=20)`

This method checks whether the path directly in front of the robot is clear of obstacles, based on the front ultrasonic sensor.  

- **Function Name**: `is_path_clear`  

- **Parameters**:  
  - `min_distance` *(int, default=20)* → Minimum safe distance in centimeters. If the detected distance is less than or equal to this threshold, the path is considered blocked.  

- **Usage**:  
  - Get the front sensor reading:  
    ```python
    front_distance = self.get_distance(self.Front_sensor)
    ```
  - If no valid reading is obtained, assume the path is blocked:  
    ```python
    if front_distance is None:
        return False
    ```
  - Compare the reading against the safe threshold:  
    ```python
    is_clear = front_distance > min_distance
    ```
  - If `debug=True`, print the path status with the measured distance:  
    ```python
    status = "CLEAR" if is_clear else "BLOCKED"
    print(f"Path status: {status} (front distance: {front_distance:.1f}cm)")
    ```
  - Return the result:  
    ```python
    return is_clear
    ```

- **Returns**:  
  - `bool` → `True` if the path is clear, `False` otherwise  


In [ ]:
def is_path_clear(self, min_distance=20):
        """
        Checks if the path ahead is clear.
        :param min_distance: Minimum safe distance in cm (default 20cm)
        :return: Boolean indicating if path is clear
        """
        front_distance = self.get_distance(self.Front_sensor)
        
        if front_distance is None:
            if self.debug:
                print("Cannot determine if path is clear - sensor error")
            return False
            
        is_clear = front_distance > min_distance
        
        if self.debug:
            status = "CLEAR" if is_clear else "BLOCKED"
            print(f"Path status: {status} (front distance: {front_distance:.1f}cm)")
            
        return is_clear

### 3.9 The Method `cleanup(self)`

This method releases all GPIO resources used by the ultrasonic sensors and resets the initialization state.

- **Function Name**: `cleanup`

- **Parameters**:  
  - *(None)* → This method does not take any parameters.  

- **Usage**:  
  - Calls `GPIO.cleanup()` to reset all GPIO pins:  
    ```python
    GPIO.cleanup()
    ```
  - Resets the internal initialization flag to allow reinitialization if needed:  
    ```python
    Ultrasonic.__init_check = False
    ```
  - If `debug=True`, prints a confirmation message:  
    ```python
    print("GPIO cleanup completed")
    ```
  - If an error occurs during cleanup, logs the exception if debug mode is enabled:  
    ```python
    print(f"Error during cleanup: {e}")
    ```


In [ ]:
def cleanup(self):
        """
        Cleans up GPIO pins and resets initialization flag.
        """
        try:
            GPIO.cleanup()
            Ultrasonic.__init_check = False
            if self.debug:
                print("GPIO cleanup completed")
        except Exception as e:
            if self.debug:
                print(f"Error during cleanup: {e}")

### 4. Test Execution Block

This block runs when the file is executed directly.  
It provides interactive testing of the `Ultrasonic` class functionality.  

- **Execution Guard**:  
    ```python
    if __name__ == "__main__":
    ```
- Ensures tests only run when the script is executed directly, not when imported as a module.

- **Usage**:

    - Initialization:
        - Creates an instance of `Ultrasonic` with `debug=True` for detailed output.
            ```python
            ultrasonic = Ultrasonic(debug=True)
            ```

    - Basic Distance Test:
        - Reads distances from `Left, Front, and Right` sensors 3 times.
        <br> Prints results or "No reading" if a sensor fails.

    - Averaged Readings Test:
        <br> Takes 5 samples from each sensor and calculates the average distance.
        ```python
        left, front, right = ultrasonic.distances(use_average=True, samples=5)
        ```

    - Obstacle Detection Test: <br>
        Uses `get_closest_obstacle()` to find the nearest object and its direction. <br>
        Prints "No obstacles detected" if no valid readings are found.<br>

    - Path Clear Test: <br>
        Uses `is_path_clear(min_distance=30)` to check if the path ahead is clear for 30 cm.<br>
        Prints "Path is clear" or "Path is blocked". <br>

    - Exception Handling: <br>
        **KeyboardInterrupt**: Stops the test gracefully when user presses *`(Ctrl+C)`* <br>
        **Other Exceptions**: Prints error details without crashing.

    - Final Cleanup: <br>
        Always calls `ultrasonic.cleanup()` if the instance exists.
        ```python
        if ultrasonic is not None:
            ultrasonic.cleanup()
        ```

    - Prints `"Test complete"` after cleanup.

In [ ]:
if __name__ == "__main__":
    ultrasonic = None
    try:
        print("Testing Improved Ultrasonic Sensor Library")
        print("=" * 50)

        ultrasonic = Ultrasonic(debug=True)

        print("\nBasic distance test...")
        for i in range(3):
            left, front, right = ultrasonic.distances()
            print(f"Reading {i+1}:")
            print(f"  Left: {left:.1f}cm" if left else "  Left: No reading")
            print(f"  Front: {front:.1f}cm" if front else "  Front: No reading")
            print(f"  Right: {right:.1f}cm" if right else "  Right: No reading")
            time.sleep(1)

        print("\nAveraged readings test...")
        left, front, right = ultrasonic.distances(use_average=True, samples=5)
        print("Averaged readings:")
        print(f"  Left: {left:.1f}cm" if left else "  Left: No reading")
        print(f"  Front: {front:.1f}cm" if front else "  Front: No reading")
        print(f"  Right: {right:.1f}cm" if right else "  Right: No reading")

        print("\nObstacle detection test...")
        distance, direction = ultrasonic.get_closest_obstacle()
        if distance:
            print(f"Closest obstacle: {distance:.1f}cm to the {direction}")
        else:
            print("No obstacles detected")

        print("\nPath clear test...")
        if ultrasonic.is_path_clear(30):
            print("Path is clear for 30cm+")
        else:
            print("Path is blocked within 30cm")

    except KeyboardInterrupt:
        print("\nTest interrupted by user")
    except Exception as e:
        print(f"\nError during test: {e}")
    finally:
        try:
            if ultrasonic is not None:
                ultrasonic.cleanup()
        except Exception:
            pass
        print("Test complete")